In [1]:
import pandas as pd

In [2]:
# read the CSV file 'number_of_reported_fatalities_by_country-year_as-of-12Dec2025.csv' into a DataFrame
path = '../data/ACLED/number_of_reported_fatalities_by_country-year_as-of-12Dec2025.csv'
df = pd.read_csv(path, encoding='utf-8', sep=';')

# display the first few rows of the DataFrame
print(df.head())

       COUNTRY  YEAR  FATALITIES
0  Afghanistan  2017       36360
1  Afghanistan  2018       42991
2  Afghanistan  2019       41419
3  Afghanistan  2020       30977
4  Afghanistan  2021       42425


In [3]:
# Extract data for Waffle Chart:
# Number of distinct countries involved in conflicts in 2025 (meaning countries with reported fatalities >= 10)
num_countries_2025 = df[(df['YEAR'] == 2025) & (df['FATALITIES'] >= 10)]['COUNTRY'].nunique()
print(f'Number of distinct countries involved in conflicts in 2025: {num_countries_2025}')

# Total number of distinct countries
total_countries = df['COUNTRY'].nunique()
print(f'Total number of distinct countries: {total_countries}')

# Calculate the percentage of countries involved in conflicts in 2025
percentage_in_conflict_2025 = (num_countries_2025 / total_countries) * 100
print(f'Percentage of countries involved in conflicts in 2025: {percentage_in_conflict_2025:.2f}%')

rounded_percentage = round(percentage_in_conflict_2025)
print(f'Rounded percentage: {rounded_percentage}%')

# Prepare data for Waffle Chart and save it to csv
waffle_data = [
    {'category': 'not_in_conflict', 'value': 100 - rounded_percentage},
    {'category': 'in_conflict', 'value': rounded_percentage},
]
waffle_df = pd.DataFrame(waffle_data)
# waffle_df.to_csv('../data/processed/waffle_chart_data.csv', index=False)

Number of distinct countries involved in conflicts in 2025: 73
Total number of distinct countries: 245
Percentage of countries involved in conflicts in 2025: 29.80%
Rounded percentage: 30%


In [4]:
waffle_df.head()

,category,value
0,not_in_conflict,70
1,in_conflict,30


In [5]:
df_2025 = df[(df['YEAR'] == 2025) & (df['FATALITIES'] >= 10)].copy()
bins = [0, 100, 1000, float('inf')]
labels = ['Low', 'Medium', 'High']
df_2025['FATALITY_CLUSTER'] = pd.cut(df_2025['FATALITIES'], bins=bins, labels=labels, right=False)
counts = df_2025['FATALITY_CLUSTER'].value_counts()
print(counts)

FATALITY_CLUSTER
Medium    27
Low       23
High      23
Name: count, dtype: int64


In [12]:
# Create a DataFrame with 'Category' in {'not_in_conflict', 'low', 'medium', 'high'} and their corresponding values in percentages rounded to whole numbers that sum to 100 and where 'not_in_conflict' is 70
cluster_counts = counts.reindex(labels, fill_value=0)
cluster_percentages = (cluster_counts / total_countries) * 100
print(cluster_percentages)
rounded_cluster_percentages = cluster_percentages.round().astype(int)
not_in_conflict_percentage = 70
waffle_data_detailed = [
    {'category': 'not_in_conflict', 'value': not_in_conflict_percentage},
    {'category': 'low', 'value': rounded_cluster_percentages['Low']},
    {'category': 'medium', 'value': rounded_cluster_percentages['Medium']},
    {'category': 'high', 'value': 100 - not_in_conflict_percentage - rounded_cluster_percentages['Low'] - rounded_cluster_percentages['Medium']},
]
waffle_df_detailed = pd.DataFrame(waffle_data_detailed)
waffle_df_detailed.head()

FATALITY_CLUSTER
Low        9.387755
Medium    11.020408
High       9.387755
Name: count, dtype: float64


,category,value
0,not_in_conflict,70
1,low,9
2,medium,11
3,high,10


In [13]:
waffle_df_detailed.to_csv('../data/processed/waffle_chart_data_detailed.csv', index=False)